# Instacart MLlib Full Run on Google Colab


- Clone repo từ GitHub.
- Chuẩn bị đủ 6 file CSV Instacart.
- Chạy `src/03_ml/local_train_mllib.py` với seed cố định.
- Lưu `reports/`, `features/`, `models/` vào Colab/Google Drive để tải về.



## 0. Runtime Notes

Khuyến nghị Colab:

- Runtime type: Python 3.
- Hardware accelerator: None cũng chạy được; High-RAM runtime tốt hơn.


In [ ]:
# ===== User-configurable constants =====
REPO_URL = "https://github.com/Zile228/InstaCart-Online-Basket-Analysis.git"
BRANCH = "AHT"

PYSPARK_VERSION = "4.1.1"

SEED = 42
SAMPLE_FRACTION = 1.0  # 1.0 = full data; 0.02 = smoke test nhanh
RUN_TAG = "colab_full_seed42" if SAMPLE_FRACTION == 1.0 else f"colab_sample_{SAMPLE_FRACTION}_seed42"

# Chọn task/model để chạy.
TASKS = "all"  # all,reorder,segmentation,basket
MODELS = "lr,rf,gbt"
K_MIN = 2
K_MAX = 8
MIN_SUPPORT = 0.003
MIN_CONFIDENCE = 0.2

# Colab full-data tuning.DRIVER_MEMORY = "8g"
SHUFFLE_PARTITIONS = 64
DEFAULT_PARALLELISM = 64
SPARK_LOCAL_DIR = "/content/spark-tmp"

print({
    "REPO_URL": REPO_URL,
    "BRANCH": BRANCH,
    "PYSPARK_VERSION": PYSPARK_VERSION,
    "SEED": SEED,
    "SAMPLE_FRACTION": SAMPLE_FRACTION,
    "RUN_TAG": RUN_TAG,
    "TASKS": TASKS,
    "MODELS": MODELS,
    "DRIVER_MEMORY": DRIVER_MEMORY,
    "SHUFFLE_PARTITIONS": SHUFFLE_PARTITIONS,
    "DEFAULT_PARALLELISM": DEFAULT_PARALLELISM,
})


## 1. Install Dependencies

Cell này cài PySpark và các thư viện Python cần cho script local. Nếu PyPI chưa có đúng `pyspark==4.1.1`, sửa `PYSPARK_VERSION` ở cell cấu hình về version khả dụng rồi chạy lại từ đầu.


In [ ]:
import os
import random
import subprocess
import sys

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

print("Python:", sys.version)
!java -version

# Install PySpark. Use a variable so the same notebook can match the repo Docker version later.
!pip -q install "pyspark=={PYSPARK_VERSION}" pandas==2.2.2 pyarrow==17.0.0 numpy==1.26.4 matplotlib==3.9.0 seaborn==0.13.2

import pyspark
print("PySpark:", pyspark.__version__)


## 2. Clone Repo

Notebook sẽ clone repo vào `/content/instacart-bigdata`.

In [ ]:
from pathlib import Path
import os
import shutil

WORKDIR = Path("/content")
REPO_DIR = WORKDIR / "instacart-bigdata"

if "<org-or-user>" in REPO_URL:
    raise ValueError("Sửa REPO_URL thành GitHub repo")

if REPO_DIR.exists():
    %cd /content/instacart-bigdata
    !git fetch --all --prune
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    %cd /content
    !git clone --branch {BRANCH} {REPO_URL} instacart-bigdata

%cd /content/instacart-bigdata
!git rev-parse --show-toplevel
!git rev-parse --short HEAD


## 3. Locate Project Root

Repo hiện tại có cấu trúc `InstaCart-Online-Basket-Analysis/src/03_ml/local_train_mllib.py`.


In [ ]:
from pathlib import Path

candidates = [
    Path("/content/instacart-bigdata/InstaCart-Online-Basket-Analysis"),
    Path("/content/instacart-bigdata"),
]

PROJECT_DIR = None
for candidate in candidates:
    if (candidate / "src/03_ml/local_train_mllib.py").exists():
        PROJECT_DIR = candidate.resolve()
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Không tìm thấy src/03_ml/local_train_mllib.py trong repo đã clone.")

print("PROJECT_DIR =", PROJECT_DIR)
print("Script =", PROJECT_DIR / "src/03_ml/local_train_mllib.py")


## 4. Prepare Data

Cần đủ 6 file CSV:

- `orders.csv`
- `order_products__prior.csv`
- `order_products__train.csv`
- `products.csv`
- `aisles.csv`
- `departments.csv`

Chọn **một** trong ba cách dưới đây.


### Option A - Kaggle API Download

Dùng nếu có `kaggle.json`. Cell sẽ yêu cầu upload file token Kaggle, tải competition data và unzip vào `PROJECT_DIR/data`.


In [ ]:
# Option A: Kaggle API. Run this cell only if you want Kaggle download.
from pathlib import Path
from google.colab import files
import os

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Upload kaggle.json when prompted...")
uploaded = files.upload()
if "kaggle.json" not in uploaded:
    raise FileNotFoundError(" chưa upload kaggle.json")

Path("/root/.kaggle").mkdir(parents=True, exist_ok=True)
Path("/root/.kaggle/kaggle.json").write_bytes(uploaded["kaggle.json"])
!chmod 600 /root/.kaggle/kaggle.json
!pip -q install kaggle

%cd {DATA_DIR}
!kaggle competitions download -c instacart-market-basket-analysis
!unzip -o -q instacart-market-basket-analysis.zip
!find . -maxdepth 2 -type f | sort


### Option B - Upload ZIP/CSV Manually

Dùng nếu  đã có các file CSV hoặc một file zip chứa 6 CSV. Với full dataset, upload ZIP thường ổn hơn upload từng CSV.


In [ ]:
# Option B: manual upload. Run this cell only if you want browser upload.
from pathlib import Path
from google.colab import files
import zipfile
import shutil

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for filename, content in uploaded.items():
    target = DATA_DIR / filename
    target.write_bytes(content)
    print("uploaded", target)
    if filename.endswith(".zip"):
        with zipfile.ZipFile(target, "r") as zf:
            zf.extractall(DATA_DIR)
        print("unzipped", target)

!find {DATA_DIR} -maxdepth 3 -type f | sort


### Option C - Google Drive

Dùng nếu dataset đã nằm trong Drive. Sửa `DRIVE_DATA_DIR` trỏ tới thư mục chứa 6 CSV, rồi copy vào `PROJECT_DIR/data`.


In [ ]:
# Option C: Google Drive. Run this cell only if data already exists in Drive.
from pathlib import Path
from google.colab import drive
import shutil

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

drive.mount("/content/drive")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/instacart")  # TODO: sửa path nếu cần

required = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]
for name in required:
    src = DRIVE_DATA_DIR / name
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, DATA_DIR / name)
    print("copied", name)


## 5. Validate Data Files

Chạy cell này sau khi hoàn tất một trong các option chuẩn bị dữ liệu.


In [ ]:
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"
required = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

missing = [name for name in required if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing files in {DATA_DIR}: {missing}")

for name in required:
    p = DATA_DIR / name
    print(f"{name:30s} {p.stat().st_size / (1024**2):10.2f} MB")


## 6. Run MLlib Pipeline

Cell này chạy `local_train_mllib.py`  repo. Các tham số seed và feature config được cố định để kết quả trên Colab/local/master tương đương về logic.

Full run dùng:

```bash
--sample-fraction 1.0
--tasks all
--models lr,rf,gbt
```

Nếu Colab thiếu RAM, chạy từng task riêng: `--tasks reorder`, `--tasks segmentation`, hoặc `--tasks basket`.


In [ ]:
from pathlib import Path
import os
import json

OUTPUT_DIR = PROJECT_DIR / "local_outputs" / RUN_TAG
FEATURE_CONFIG = PROJECT_DIR / "../notebooks/my_work/outputs/sklearn_research/selected_features_for_mllib.json"
Path(SPARK_LOCAL_DIR).mkdir(parents=True, exist_ok=True)

# If repo is cloned as the project subdir only, use a fallback config path.
if not FEATURE_CONFIG.exists():
    fallback = PROJECT_DIR / "notebooks/my_work/outputs/sklearn_research/selected_features_for_mllib.json"
    FEATURE_CONFIG = fallback if fallback.exists() else None

# If the selected-feature file is not present in the cloned repo, recreate the same config inline.
# This keeps Colab/local/master runs aligned with the feature set used in the report work.
if FEATURE_CONFIG is None:
    generated_config = PROJECT_DIR / "local_outputs" / "selected_features_for_mllib.generated.json"
    generated_config.parent.mkdir(parents=True, exist_ok=True)
    generated_config.write_text(json.dumps({
        "source": "generated by 05_colab_full_mllib_run.ipynb",
        "sample_frac": None,
        "best_model": {
            "trial": "hgb_lr0.05_iter160_leaf31",
            "model": "hist_gbdt",
            "average_precision": 0.37601770317532923,
            "roc_auc": 0.8206271797102557,
            "precision_pos": 0.2424372914199591,
            "recall_pos": 0.704630788485607,
            "f1_pos": 0.36075290348418104,
            "recall_at_10": 0.5764791843145919,
            "recall_at_20": 0.7516188626816104
        },
        "reorder_numeric_features": [
            "up_orders_since_last",
            "up_order_count",
            "up_reorder_rate",
            "up_order_rate_since_first",
            "u_reorder_rate",
            "p_reorder_rate",
            "u_avg_basket_size",
            "u_std_days_since_prior",
            "p_aisle_id",
            "u_dairy_ratio",
            "p_total_orders",
            "u_unique_departments",
            "u_total_items",
            "u_avg_days_since_prior",
            "u_preferred_hour",
            "up_avg_position",
            "up_first_order_number",
            "u_unique_aisles"
        ],
        "reorder_categorical_features": ["p_department_id"],
        "segmentation_features": [
            "recency",
            "frequency",
            "volume",
            "u_reorder_rate",
            "u_organic_ratio",
            "u_distinct_products",
            "u_unique_departments",
            "u_produce_ratio",
            "u_dairy_ratio"
        ],
        "basket": {
            "algorithm": "FPGrowth",
            "sort_rules_by": ["lift", "confidence", "support"],
            "starting_grid": {
                "minSupport": [0.001, 0.003, 0.005, 0.01],
                "minConfidence": [0.1, 0.2, 0.3]
            }
        }
    }, indent=2), encoding="utf-8")
    FEATURE_CONFIG = generated_config

cmd = [
    "python3", str(PROJECT_DIR / "src/03_ml/local_train_mllib.py"),
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--master", "local[*]",
    "--tasks", TASKS,
    "--models", MODELS,
    "--sample-fraction", str(SAMPLE_FRACTION),
    "--seed", str(SEED),
    "--k-min", str(K_MIN),
    "--k-max", str(K_MAX),
    "--min-support", str(MIN_SUPPORT),
    "--min-confidence", str(MIN_CONFIDENCE),
    "--driver-memory", DRIVER_MEMORY,
    "--shuffle-partitions", str(SHUFFLE_PARTITIONS),
    "--default-parallelism", str(DEFAULT_PARALLELISM),
    "--local-dir", SPARK_LOCAL_DIR,
    "--overwrite",
]
cmd.extend(["--feature-config", str(FEATURE_CONFIG)])

print("OUTPUT_DIR =", OUTPUT_DIR)
print("FEATURE_CONFIG =", FEATURE_CONFIG)
print("Command:")
print(" ".join(cmd))

# Run in notebook so logs are visible.
import subprocess, time
start = time.time()
completed = subprocess.run(cmd, cwd=str(PROJECT_DIR), text=True)
elapsed = time.time() - start
print("Exit code:", completed.returncode)
print(f"Elapsed: {elapsed/60:.2f} min")
if completed.returncode != 0:
    raise RuntimeError("local_train_mllib.py failed. Check logs above.")


## 7. Inspect Reports

Cell này đọc lại JSON report chính và in các số liệu cần đưa vào báo cáo.


In [ ]:
import json
from pathlib import Path

report_dir = OUTPUT_DIR / "reports"
summary_path = report_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(summary_path)

summary = json.loads(summary_path.read_text())
print(json.dumps(summary.keys().__repr__(), indent=2))

if "reorder" in summary:
    best = summary["reorder"]["best_model"]
    print("\nReorder best model")
    for key in ["model", "auc_pr", "auc_roc", "accuracy", "precision_pos", "recall_pos", "f1_pos", "tp", "fp", "fn", "tn"]:
        print(f"  {key}: {best.get(key)}")

if "segmentation" in summary:
    seg = summary["segmentation"]
    print("\nSegmentation")
    print("  user_count:", seg.get("user_count"))
    print("  best_k:", seg.get("best_k"))
    print("  best_silhouette:", seg.get("best_silhouette"))

if "basket" in summary:
    basket = summary["basket"]
    print("\nBasket")
    for key in ["basket_count", "freq_itemset_count", "rule_count", "min_support", "min_confidence"]:
        print(f"  {key}: {basket.get(key)}")
    print("\nTop 5 rules:")
    for rule in basket.get("top_rules", [])[:5]:
        print("  ", rule["antecedent_names"], "=>", rule["consequent_names"], "conf=", round(rule["confidence"], 4), "lift=", round(rule["lift"], 4))


## 8. Save Results

Cell này zip output để tải về. Nếu muốn lưu vào Drive, mount Drive ở cell trước rồi copy file zip vào `MyDrive`.


In [ ]:
from pathlib import Path
from google.colab import files
import shutil

zip_base = Path("/content") / RUN_TAG
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUTPUT_DIR))
print("ZIP:", zip_path)
print("Size MB:", Path(zip_path).stat().st_size / (1024**2))

files.download(zip_path)


## 9. Optional: Copy Results to Google Drive


In [ ]:
# Optional: copy zip to Drive.
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")
drive_out = Path("/content/drive/MyDrive/instacart_mllib_outputs")
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, drive_out / Path(zip_path).name)
print("Copied to", drive_out / Path(zip_path).name)


## 10. Later: Re-run on Master Cluster

Khi master mở lại, không upload trực tiếp kết quả Colab để thay thế kết quả chính thức. Nên chạy lại các script HDFS chính để đồng bộ môi trường Docker/Spark  nhóm:

```bash
spark-submit /home/nhom05/work/01_preprocessing/02_feature_engineering.py
spark-submit /home/nhom05/work/03_ml/01_reorder_classifier.py
spark-submit /home/nhom05/work/03_ml/02_customer_segmentation.py
spark-submit /home/nhom05/work/03_ml/03_market_basket_fpgrowth.py
spark-submit /home/nhom05/work/03_ml/04_aisle_pca_kmeans.py
```

Dùng report Colab để:

- Viết trước chương kết quả.
- Kiểm tra logic mô hình/feature trước khi merge.
- So sánh sanity check với kết quả master sau này.
